### unit: eV, K, ps, Å, $\overset{\circ}{\text{A}}$

In [4]:
import numpy as np
import read_parameters
import its

def enhanced_sampling(enhanced_sampling_method, if_initial,\
                      simulation_temperature, simulation_maxsteps,\
                      time_step, potential_energy, md_force, log_morest,\
                      parameter_file='MoREST.parameter'):
    '''
    This API function is called to excute enhanced sampling by MD/MC module.
    --------
    INPUT:
        enhanced_sampling_method: Specify the method will be used, e.g., ITS, REMD...
        if_initial:               Specify whether this API is called for the first time.
        simulation_temperature:   The temperature used in MD/MC module. Unit: K.
        simulation_maxsteps:      The simulation length, which is specified in number of steps, in MD/MC module.
        time_step:                The time step in MD/MC module. Unit: ps.
        potential_energy:         The potential energy of current configuration from MD/MC module.
        md_force:                 The force vector of current configuration from MD/MC module.
        log_morest:               The path and name of the log file for MoREST.
        parameter_file:           The path and name of the parameter file.
    OUTPUT:
        its_bias_force:           The bias force vector generated by ITS and returned to MD/MC module. 
    '''
    
    if enhanced_sampling_method in ['ITS','its']:
        if if_initial:
            # from read_parameters import read_parameters
            its_parameters = read_parameters.read_parameters(log_morest,parameter_file).get_its_parameters()
            #print(its_parameters)
            # from its import its_optimization. return bias_force, {p_k}, {n_k}
            bias_force, p_k, n_k = its.its().its_optimization()
            return bias_force
        else:
            # from its import its_if_converge
            if its.its_if_converge():
                # from its import import its_sampling
                bias_force = its.its_sampling() 
                return bias_force
            else:
                # from its import its_optimization
                bias_force, p_k, n_k = its.its_optmization()
                return bias_force
    else:
        log_morest.write('No enhanced sampling method was matched.\n')
        log_morest.close()
        return np.array([0])

In [5]:
log_morest = open('MoREST.log','w')
simulation_temperature = 798 # K
simulation_maxsteps = 10000
time_step = 0.0001 # ps
potential_energy = 4000 # eV
md_force = np.array([[1.,2.,3.],[1.,3.,2.],[3.,1.,2.],[3.,2.,1.],[1.,2.,3.],[1.,3.,2.],[3.,1.,2.],[3.,2.,1.]])
enhanced_sampling('its', True, simulation_temperature, simulation_maxsteps,\
                  time_step, potential_energy, md_force, log_morest)

{'its_lower_bound_temperature': 700.0, 'its_upper_bound_temperature': 900.0, 'its_number_of_replica': 10, 'its_replica_arrange': 0.0, 'its_trial_MD_steps': 1000, 'its_delta_pk': 1e-05, 'its_replica_temperature': array([700.        , 719.82214482, 740.20560024, 761.1662611 ,
       782.72047233, 804.88504169, 827.67725291, 851.11487916,
       875.21619686, 900.        ]), 'its_initial_nk': array([39.80426259, 25.21551738, 16.1758022 , 10.50444399,  6.90308792,
        4.58917537,  3.08537924,  2.09715556,  1.4406888 ,  1.        ]), 'its_pk0': array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])}


AttributeError: module 'its' has no attribute 'its_optimization'

In [9]:
its_para = np.load('MoREST_ITS_parameters.npy',allow_pickle=True).item()
print(its_para['its_lower_bound_temperature'])

700.0


In [18]:
for key in its_para:
    print(key,type(key))

its_lower_bound_temperature <class 'str'>
its_upper_bound_temperature <class 'str'>
its_number_of_replica <class 'str'>
its_replica_arrange <class 'str'>
its_trial_MD_steps <class 'str'>
its_delta_pk <class 'str'>
its_replica_temperature <class 'str'>
its_initial_nk <class 'str'>
its_pk0 <class 'str'>
